**STEAM MACHINE LEARNING**

**Integrantes**:

João Pedro Costa Cruz

João Vitor Souza Tavares

Lucas Antônio Araújo Santos

**Link para o dataset original**:

https://www.kaggle.com/datasets/fronkongames/steam-games-dataset

**Descrição do problema**:

O dataset escolhido traz informações sobre jogos virtuais da plataforma Steam. A ideia é determinar se um jogo é um "Sucesso". Um jogo é considerado um sucesso se ele tiver uma taxa de aprovação de reviews positivos maior ou igual a 80% e ao menos 100 reviews.

O problema foi estruturado como uma tarefa de **Classificação** porque o nosso objetivo final é mapear os dados de entrada para uma decisão categórica e discreta (Sucesso vs. Não Sucesso). Isso permite que tomadores de decisão (como publicadoras de jogos) façam avaliações qualitativas e utilizem métricas de diagnóstico de erro cruciais (como a taxa de falsos positivos e falsos negativos através de matrizes de confusão) para minimizar os riscos de investimento.

In [ ]:
#==================================
#   IMPORTAÇÃO DAS BIBLIOTECAS
#==================================

# Manipulação dos dados
import pandas as pd
import numpy as np

# Visualização gráfica
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações do ambiente de plotagem
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Ignorar avisos de atualizações de bibliotecas
import warnings
warnings.filterwarnings('ignore')

print("Configurações do ambiente concluídas.")

In [ ]:
#============================================
#   DATASET E DEFINIÇÃO DO ATRIBUTO-ALVO
#============================================

# URL do dataset
url_dataset = "https://raw.githubusercontent.com/LucasAAraujoS/steam-machine-learning/refs/heads/main/steam_games_clean.csv"

# Carrega o arquivo usando o ponto e vírgula (;) como delimitador
df = pd.read_csv(url_dataset, sep=';')

#----------------------------------------------------------------------------------------------------------------------------

# Cálculo do volume total de avaliações (reviews)
df['Total_Reviews'] = df['Positive'] + df['Negative']

# Cálculo da taxa de aprovação de um jogo
df['Approval_Rate'] = np.where(df['Total_Reviews'] > 0, df['Positive'] / df['Total_Reviews'], 0.0)

# Definição do atributo-alvo
df['Success'] = ((df['Approval_Rate'] >= 0.80) & (df['Total_Reviews'] >= 100)).astype(int)

print(f"Dataset carregado com sucesso!")
print(f"Dimensões do DataFrame: {df.shape[0]} linhas e {df.shape[1]} colunas.")

## 1. Engenharia do Alvo e Estatística Descritiva

Esta seção cobre: distribuição do atributo-alvo (`Success`), estatística descritiva geral do dataset e remoção das colunas usadas para construir o alvo (evitando data leakage).

In [ ]:
#==================================
#   DISTRIBUIÇÃO DO ATRIBUTO-ALVO
#==================================

# Calculo da frequ~encia dos dados
success_counts = df['Success'].value_counts().sort_index()
success_pct = df['Success'].value_counts(normalize=True).sort_index() * 100

print("Contagem por classe:")
print(success_counts)
print("\nProporção por classe (%):")
print(success_pct.round(2))
print("\n")

# Plotagem do gráfico
fig, ax = plt.subplots()
sns.countplot(x='Success', data=df, palette=['#c0392b', '#27ae60'], ax=ax) # Desenha o gráfico
ax.set_xticklabels(['Não Sucesso (0)', 'Sucesso (1)']) # Altera os nomes de "0" e "1 para o que está dentro dos colchetes
ax.set_title('Distribuição da variável alvo: Sucesso vs. Não Sucesso')
ax.set_xlabel('')
ax.set_ylabel('Quantidade de jogos')
for i, v in enumerate(success_counts):
    ax.text(i, v + 1500, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretação:**

O dataset apresenta um forte desbalanceamento de classes: **89,56% dos jogos (112.718) são classificados como "Não Sucesso"**, enquanto apenas **10,44% (13.137) são "Sucesso"**, ou seja, uma proporção aproximada de 9 para 1.

Isso reflete uma realidade do mercado da Steam: a grande maioria dos jogos publicados na plataforma não atinge um patamar consistente de avaliações positivas somado a um volume relevante de reviews — o que é coerente com o fato de a Steam ser uma plataforma aberta, com milhares de lançamentos pequenos, indie ou pouco divulgados.

Esse desbalanceamento é uma informação crítica para as próximas etapas do projeto: ele justifica (a) o uso de `stratify=y` na separação treino/teste para preservar essa proporção em ambos os conjuntos, e (b) a priorização do F1-Score e da matriz de confusão em vez da acurácia simples na avaliação dos modelos, já que um classificador ingênuo que sempre previsse "Não Sucesso" já acertaria quase 90% dos casos sem aprender nada de útil.

### Estatística descritiva geral do dataset

In [ ]:
#==================================
#   ESTATÍSTICA DESCRITIVA GERAL
#==================================

print(f"O dataset possui {df.shape[0]:,} registros e {df.shape[1]} atributos.\n")

print("Tipos de dados por coluna:")
print(df.dtypes)

valores_ausentes = df.isnull().sum()
print("\nValores ausentes por coluna (apenas colunas com ausências):")
print(valores_ausentes[valores_ausentes > 0])

print(f"\nLinhas totalmente duplicadas: {df.duplicated().sum()}")
print(f"Nomes de jogos duplicados (podem ser DLCs/remasters/reentradas): {df['Name'].duplicated().sum()}")

df[['Price', 'Positive', 'Negative', 'Achievements']].describe()

**Interpretação:**

O dataset contém 125.855 jogos e 14 atributos, com um volume ausente concentrado em variáveis textuais/categóricas: `Developers` (8.449), `Publishers` (8.922), `Categories` (8.966) e `Genres` (8.423) — em torno de 6,7% das linhas não têm gênero ou desenvolvedor informado, o que provavelmente reflete jogos cadastrados de forma incompleta na Steam (ex: *playtests* ou títulos removidos). Há também 18 linhas totalmente duplicadas e 1.210 nomes repetidos, que precisarão ser investigados na Etapa 2 (podem ser duplicatas reais ou apenas jogos com o mesmo nome).

Quanto ao preço, `a média (US$ 4,81) é bem maior que a mediana (US$ 2,39)`, o que indica uma distribuição assimétrica à direita: a maioria dos jogos é barata, mas alguns títulos caros (até US$ 999,98) puxam a média para cima. O mesmo padrão, ainda mais extremo, aparece em `Positive` e `Negative`: a mediana de reviews positivas é de apenas 4, mas a média passa de 1.000, evidenciando a presença de outliers fortes — alguns poucos jogos com milhões de reviews (blockbusters) distorcem completamente a média em relação à realidade da maioria dos títulos da plataforma.

### Remoção das colunas usadas para construir o alvo (evitando *data leakage*)

In [ ]:
#===================================================
#   REMOÇÃO DAS COLUNAS DE VAZAMENTO (DATA LEAKAGE)
#===================================================

# Positive, Negative, Total_Reviews e Approval_Rate foram usadas para CALCULAR
# o alvo (Success). Mantê-las como atributos preditivos permitiria ao modelo
# 'colar' a resposta em vez de aprender padrões reais.

# Mantemos 'df' completo para a análise exploratória e criamos
# 'df_model' -- sem essas colunas -- para ser o ponto de partida do
# que vem a seguir (pré-processamento e modelagem).

leakage_cols = ['Positive', 'Negative', 'Total_Reviews', 'Approval_Rate']

df_model = df.drop(columns=leakage_cols)

print("Colunas removidas para evitar data leakage:", leakage_cols)
print(f"\nColunas restantes para modelagem ({df_model.shape[1]}):")
print(list(df_model.columns))

**Interpretação:**

`Positive`, `Negative`, `Total_Reviews` e `Approval_Rate` não podem seguir para o treinamento porque foram exatamente as variáveis usadas para definir `Success`: um modelo treinado com elas não estaria aprendendo a prever sucesso a partir de características do jogo (preço, gênero, plataforma etc.), e sim reconstruindo uma fórmula que já conhecemos — isso é chamado de vazamento de dados (*data leakage*) e infla artificialmente as métricas sem gerar um modelo útil na prática.

`df_model` guarda a versão do dataset já livre desse problema, pronta para ser usada. `df` (com todas as colunas originais) continua disponível para as análises de correlação e distribuição que vêm a seguir.